This notebook is used to complete question 2 of the homework.

(i) First, we initialise the drift-diffusion model (DDM) as per the lecture slides.

(ii) We then hyperparameter tune values of drift, $v$, between 0.5 - 1.5 and record the mean differences of the two decision boundaries.

(iii) Finally, we then hyperparameter tune other factors to determine the impact on the mean differences.

# (i) Initialisation...

In [1]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def ddm_trial(v, a, beta, tau, dt=1e-3, scale=1.0, max_time=10.):
    """
    Simulates one realization of the diffusion process given
    a set of parameters and a step size `dt`.

    Parameters:
    -----------
    v     : float
        The drift rate (rate of information uptake)
    a     : float
        The boundary separation (decision threshold).
    beta  : float in [0, 1]
        Relative starting point (prior option preferences)
    tau   : float
        Non-decision time (additive constant)
    dt    : float, optional (default: 1e-3 = 0.001)
        The step size for the Euler algorithm.
    scale : float, optional (default: 1.0)
        The scale (sqrt(var)) of the Wiener process. Not considered
        a parameter and typically fixed to either 1.0 or 0.1.
    max_time: float, optional (default: .10)
        The maximum number of seconds before forced termination.

    Returns:
    --------
    (x, c) - a tuple of response time (y - float) and a 
        binary decision (c - int) 
    """

    # initialisation...
    evidence_acc = a * beta
    time_acc = 0.


    # simulation...
    while (time_acc < max_time):
        # with each timestamp, we update the evidence accumulation by dX_t, 
        #  which we use the Euler-Maruyama method as described in slide17...
        
        # Z_t ~ N(0,1)
        Z_t = np.random.randn()

        evidence_acc += v*dt + scale*np.sqrt(dt)*Z_t  
        time_acc += dt


        # checking if either boundary was reached...
        if (evidence_acc >= a):
            return (time_acc + tau, 1) 
        
        if (evidence_acc <= 0):
            return (time_acc + tau, 0)
        

    # return the closest boundary if we exceed the time limit...
    midpt = a / 2
    bin_decision = 1 if evidence_acc > midpt else 0

    return (max_time + tau, bin_decision)
    

def ddm(num_sims, v, a, beta, tau, dt=1e-3, scale=1.0, max_time=10.):
    data = np.zeros((num_sims, 2))
    for n in range(num_sims):
        data[n, :] = ddm_trial(v, a, beta, tau, dt, scale, max_time)
    return data


def visualize_diffusion_model(data, figsize=(8, 6)):
    f, ax = plt.subplots(1, 1, figsize=figsize)
    sns.histplot(data[:, 0][data[:, 1] == 1], color='maroon', alpha=0.7, ax=ax, label='Correct responses')
    sns.histplot(data[:, 0][data[:, 1] == 0], color='gray', ax=ax, label='Incorrect responses')
    sns.despine(ax=ax)
    ax.set_xlabel('Response time (s)', fontsize=18)
    ax.set_ylabel('')
    ax.legend(fontsize=18)
    ax.set_yticks([])
    return f

# (ii) hyperparamter tuning on $v$ and analysis on mean difference...

In [26]:
v_range = np.linspace(0.5, 1.5, 25)
N = 2000


# other parameters are fixed such that the process doesn't just reach the upper boundary...
a = 0.5
beta = 0.4
tau = 0.7


if True:
    upperbound_mean = np.zeros(len(v_range))
    lowerbound_mean = np.zeros(len(v_range))
    diff_mean = np.zeros(len(v_range))


    for i, v in enumerate(v_range):
        ddm_data = ddm(N, v, a, beta, tau)


        # separating trial outcomes into respective boundaries... 
        #   (we just want the response time so only grab that column)
        upperbound = ddm_data[ddm_data[:, 1] == 1, 0]
        lowerbound = ddm_data[ddm_data[:, 1] == 0, 0]


        # computing respective means...
        upperbound_mean[i] = np.mean(upperbound) if len(upperbound) > 0 else np.nan
        lowerbound_mean[i] = np.mean(lowerbound) if len(lowerbound) > 0 else np.nan

        diff_mean[i] = upperbound_mean[i] - lowerbound_mean[i]

else:
    # testing to make sure that the process reaches both boundaries relatively equally...
    v = 1.5
    a = 0.5
    beta = 0.5
    tau = 0.7

    ddm_data = ddm(N, v, a, beta, tau)

    upper_rts = ddm_data[ddm_data[:, 1] == 1, 0]
    lower_rts = ddm_data[ddm_data[:, 1] == 0, 0]

    print("n upper:", len(upper_rts))
    print("n lower:", len(lower_rts))
    print("mean upper:", np.mean(upper_rts))
    print("mean lower:", np.mean(lower_rts))
    print("sd upper:", np.std(upper_rts))
    print("sd lower:", np.std(lower_rts))
    


In [27]:
diff_mean

array([0.01581845, 0.01352449, 0.01674116, 0.01505579, 0.0150997 ,
       0.01856524, 0.01742904, 0.01816497, 0.01788334, 0.0166729 ,
       0.01978853, 0.01692361, 0.01762402, 0.01421323, 0.01397556,
       0.01699566, 0.01659943, 0.01956215, 0.01499479, 0.01436241,
       0.01765144, 0.01691036, 0.01660115, 0.02265637, 0.01489376])

interpretation:

values more or less consistently around 0.015 - 0.020 and does not change with v vals
also the diff mean_i > 0 --> upperbound mean consistently slower than lowerbound marginally

In [28]:
# f = visualize_diffusion_model(ddm_data)

# (iii) hyperparameter tuning on all variables and analysis on means and stdevs...

to do:

add stdev to v,
then do linspace on lpha, beta, tau

then report all respective results and analyse